# Post-`create_dataset` cleanup

This notebook does two things, in order, for a subset folder produced by `mdc.create_dataset` :

1. **Join step** — reads `image_metadata.json`, finds which `case_id`s actually have a usable image (by default: `image_subtype == 'skin_photograph'`), and extracts only those rows from `cases.csv` into `cases_with_image.csv`.
2. **False-positive triage step** — scans the case text in `cases_with_image.csv` for signs that the target disease term was mentioned only to be *ruled out* (negative test, excluded differential, etc.), and splits the result into `cases_clean.csv` (likely genuine) and `cases_flagged_for_review.csv` (needs a human look).

Only edit the **CONFIG** cell below to point this at a different disease subset.

In [1]:
# ===================== CONFIG =====================

import json
import re
from pathlib import Path

import pandas as pd

# MODIFY THE FILEPATH HERE
SUBSET_DIR = Path("medical_datasets/malaria")

CASES_CSV = SUBSET_DIR / "cases.csv"
IMAGE_METADATA_JSON = SUBSET_DIR / "image_metadata.json"

CASES_WITH_IMAGE_CSV = SUBSET_DIR / "cases_with_image.csv"
CASES_CLEAN_CSV = SUBSET_DIR / "cases_clean.csv"
CASES_FLAGGED_CSV = SUBSET_DIR / "cases_flagged_for_review.csv"

# Which image subtype(s) count as "usable" for this project
VALID_IMAGE_SUBTYPES = {"skin_photograph"}

# Disease term(s) to check for negation around. Add synonyms as needed.
LEPROSY_SKIN_KEYWORDS = [
    'leprosy',
    'lepromatous leprosy',
    'borderline lepromatous leprosy',
    'tuberculoid leprosy',
    'multibacillary leprosy',
    'nodular leprosy',
    'leprosy reaction',
]


NEGATION_WINDOW_CHARS = 250

NEGATION_CUES = [
    "negative", "ruled out", "rule out", "excluded", "not detected",
    "non-reactive", "nonreactive", "unremarkable", "denied", "no evidence of",
    "were all normal", "came back negative", "tested negative",
]

print(f"Subset dir: {SUBSET_DIR.resolve()}")

Subset dir: /home/tokuden/VGU_WS26_ClinicalProject_BHTBH/Demos/medical_datasets/malaria


## Step 1 — Join `cases.csv` with `image_metadata.json`

`image_metadata.json` is a list of image records, each with a `case_id` and an `image_subtype`. We collect the set of `case_id`s that have at least one image of a valid subtype, then keep only those rows of `cases.csv`.

In [2]:
with open(IMAGE_METADATA_JSON, "r", encoding="utf-8") as f:
    image_records = [json.loads(line) for line in f if line.strip()]


print(f"Total image records: {len(image_records)}")

# Sanity check: what image_subtypes actually exist in this file?
subtype_counts = pd.Series([r.get("image_subtype") for r in image_records]).value_counts(dropna=False)
print("\nimage_subtype breakdown:")
print(subtype_counts)

Total image records: 76

image_subtype breakdown:
ct                          25
immunostaining              12
giemsa                       9
h&e                          8
mri                          5
fundus_photograph            5
other_medical_photograph     4
skin_photograph              4
ultrasound                   2
gram                         1
pas                          1
Name: count, dtype: int64


In [4]:
valid_case_ids = {
    r["case_id"]
    for r in image_records
}

print(f"Unique case_ids with a valid image: {len(valid_case_ids)}")

Unique case_ids with a valid image: 19


In [5]:
cases_df = pd.read_csv(CASES_CSV)
print(f"Total rows in cases.csv: {len(cases_df)}")

assert "case_id" in cases_df.columns, "cases.csv must have a case_id column"

cases_with_image_df = cases_df[cases_df["case_id"].isin(valid_case_ids)].copy()
print(f"Rows with a matching usable image: {len(cases_with_image_df)}")

cases_with_image_df.to_csv(CASES_WITH_IMAGE_CSV, index=False)
print(f"Saved -> {CASES_WITH_IMAGE_CSV}")

Total rows in cases.csv: 45
Rows with a matching usable image: 19
Saved -> medical_datasets/malaria/cases_with_image.csv


## Step 2 — Flag likely false positives (disease mentioned but negated)

For each case, find every occurrence of a disease term and look at a window of text around it. If a negation cue ("negative", "ruled out", "excluded", etc.) appears in that window, the case is flagged for manual review rather than auto-dropped -- this catches things like:

> "Serum tests for pathogens including ... Dengue virus ... were all negative."

This is a heuristic, not a guarantee -- it will have some false positives of its own (e.g. "fever was not accompanied by rash; dengue was confirmed"), which is why flagged cases go to a separate CSV for a quick human look rather than being deleted outright.

In [5]:
def find_negation_hits(text: str, disease_terms, negation_cues, window: int):
    """Return a list of (matched_term, snippet) for every disease-term occurrence
    that has a negation cue within `window` characters on either side."""
    if not isinstance(text, str):
        return []

    text_lower = text.lower()
    hits = []

    for term in disease_terms:
        for match in re.finditer(re.escape(term.lower()), text_lower):
            start = max(0, match.start() - window)
            end = min(len(text_lower), match.end() + window)
            snippet = text_lower[start:end]

            if any(cue in snippet for cue in negation_cues):
                hits.append((term, text[start:end].strip()))

    return hits


def is_likely_negated(text: str) -> bool:
    return len(find_negation_hits(text, LEPROSY_SKIN_KEYWORDS, NEGATION_CUES, NEGATION_WINDOW_CHARS)) > 0

In [6]:
assert "case_text" in cases_with_image_df.columns, "cases.csv must have a case_text column"

cases_with_image_df["likely_negated"] = cases_with_image_df["case_text"].apply(is_likely_negated)

n_flagged = cases_with_image_df["likely_negated"].sum()
n_total = len(cases_with_image_df)
print(f"Flagged {n_flagged} / {n_total} cases ({n_flagged / n_total:.1%}) as likely false positives")

Flagged 8 / 22 cases (36.4%) as likely false positives


In [7]:
cases_clean_df = cases_with_image_df[~cases_with_image_df["likely_negated"]].drop(columns=["likely_negated"])
cases_flagged_df = cases_with_image_df[cases_with_image_df["likely_negated"]].drop(columns=["likely_negated"])

cases_clean_df.to_csv(CASES_CLEAN_CSV, index=False)
cases_flagged_df.to_csv(CASES_FLAGGED_CSV, index=False)

print(f"Clean cases   -> {CASES_CLEAN_CSV} ({len(cases_clean_df)} rows)")
print(f"Flagged cases -> {CASES_FLAGGED_CSV} ({len(cases_flagged_df)} rows, review these manually)")

Clean cases   -> medical_datasets/leprosy/cases_clean.csv (14 rows)
Flagged cases -> medical_datasets/leprosy/cases_flagged_for_review.csv (8 rows, review these manually)


## Optional — preview why each flagged case was caught

Run this to print the actual negation snippet found for each flagged case, so you can quickly eyeball whether it's a true negation (drop) or a false alarm from this heuristic (keep, move back into `cases_clean.csv` manually).

In [ ]:
for _, row in cases_flagged_df.iterrows():
    hits = find_negation_hits(row["case_text"], DISEASE_TERMS, NEGATION_CUES, NEGATION_WINDOW_CHARS)
    print(f"\n=== {row['case_id']} ===")
    for term, snippet in hits:
        print(f"  matched term: '{term}'")
        print(f"  ...{snippet}...")